# xarray

> Utilities for handling Euclid FITS images as xarray Datasets, using dask for efficient computation and memory usage.

In [ ]:
# | default_exp euclid.xarray

In [ ]:
# | export

import json
from warnings import catch_warnings, filterwarnings

import numpy as np
import pandas as pd
import xarray as xr
from astropy.io import fits
from kerchunk.combine import MultiZarrToZarr
from kerchunk.fits import process_file
from nicl.euclid.utilities import (
    get_dither_id_from_filename,
    get_filter_from_filename,
    get_obs_id_from_filename,
    get_tile_index_from_filename,
)

ImportError: cannot import name 'get_dither_id_from_filename' from 'nicl.euclid.utilities' (/Users/spb/repos/IntraClusterLight/nicl/nicl/euclid/utilities.py)

In [ ]:
# | hide
# # Additional imports used in the examples

from nicl.euclid.utilities import default_data_path, get_nisp_images_for_observation

In [ ]:
# | exporti

def _fix_byte_order(refs):
    """Correctly identify the byte order as bigendian.

    This is a workaround. A proper fix has been submitted via this PR:
    https://github.com/fsspec/kerchunk/pull/531
    When that PR is merged this function will be unnecessary, but not cause any harm.
    """
    for k in refs.keys():
        if "zarray" in k:
            z = json.loads(refs[k])
            z["dtype"] = np.dtype(z["dtype"]).newbyteorder(">").str
            refs[k] = json.dumps(z).encode()
    return refs

In [ ]:
# | exporti

def _rename(refs, new_name):
    """Rename the zarray in a kerchunk json reference."""
    new_refs = {}
    for key in refs.keys():
        if "/" in key:
            parts = key[key.find("/")+1:]
            new_key = f"{new_name}/{parts}"
            new_refs[new_key] = refs[key]
    return new_refs

In [ ]:
# | exporti

def _get_ext_type(x):
        return x.split(".")[1] if "." in x else None

def _get_ext_det(x):
        return x.split(".")[0] if "." in x else x

In [ ]:
# | export

def create_zarr_ref_from_fits(fns, return_wcs=False):
    """Create a zarr reference from a set of Euclid fits files.

    This uses kerchunk to build a zarr reference file for accessing data directly from fits files.
    The resulting reference can then be opened as a xarray Dataset with `open_zarr_ref_as_dataset`.
    """
    if isinstance(fns, str):
        fns = [fns]
    fn = fns[0]
    if "MER" in str(fn):
        product_id_name = "tile_index"
        get_product_id = get_tile_index_from_filename
    else:
        product_id_name = "observation_id"
        get_product_id = get_obs_id_from_filename
    with fits.open(fn) as hdul:
        exts = list(h.name for h in hdul if h.size > 0)
    ref_files = []
    product_ids = []
    dither_ids = []
    filters = []
    wcs_list = []
    for fn in fns:
        product_id = get_product_id(fn)
        product_ids.append(product_id)
        dither_id = get_dither_id_from_filename(fn)
        if dither_id is not None:
            dither_ids.append(dither_id)
        filter = get_filter_from_filename(fn)
        if dither_id is not None:
            filters.append(filter)
        print(fn, product_id, dither_id, filter)
        ref_exts = []
        coords = []
        coord_name = "extension"
        for ext in exts:
            ext_type = _get_ext_type(ext)
            out = process_file(fn, extension=ext)
            if ext_type is not None:
                out = _rename(out, ext_type)
                coord_name = "detector"
            out = _fix_byte_order(out)
            ref_exts.append(out)
            coord = _get_ext_det(ext)
            coords.append(coord)
            hdr = fits.getheader(fn, ext)
            if return_wcs:
                wcs = {k: hdr[k] for k in ("CTYPE1", "CTYPE2",
                                           "CD1_1", "CD1_2", "CD2_1", "CD2_2",
                                           "CRPIX1", "CRPIX2", "CRVAL1", "CRVAL2",
                                           "NAXIS1", "NAXIS2")}
                wcs[coord_name] = coord
                if dither_id is not None:
                    wcs["dither"] = dither_id
                if dither_id is not None:
                    wcs[product_id_name] = product_id
                wcs_list.append(wcs)
        mzz = MultiZarrToZarr(
            ref_exts,
            coo_map = {coord_name: coords},
            concat_dims=[coord_name],
            identical_dims = ['x', 'y']
        )
        ref_exts = mzz.translate()
        ref_files.append(ref_exts)
    coo_map = {}
    concat_dims = []
    if len(np.unique(product_ids)) > 0:
        coo_map[product_id_name] = product_ids
        concat_dims.append(product_id_name)
    if len(np.unique(dither_ids)) > 0:
        coo_map["dither"] = dither_ids
        concat_dims.append("dither")
    if len(np.unique(filters)) > 0:
        coo_map["filter"] = filters
        concat_dims.append("filter")
    mzz = MultiZarrToZarr(
        ref_files,
        coo_map = coo_map,
        concat_dims=concat_dims,
        identical_dims = ['x', 'y']
    )
    with catch_warnings():
        filterwarnings("ignore", "Concatenated coordinate .* contains less than expected")
        out = mzz.translate()
    if return_wcs:
        wcs = pd.DataFrame(wcs_list)
        wcs = wcs.groupby([coord_name, "dither", product_id_name]).first()
        wcs = wcs.to_xarray()
        return out, wcs
    else:
        return out

In [ ]:
# | export

def open_zarr_ref_as_dataset(ref):
    """Open a zarr reference as a dask xarray Dataset."""
    ds = xr.open_dataset(
        "reference://", engine="zarr",
        chunks='auto',
        backend_kwargs={
            "storage_options": {
                "fo": ref,
            },
            "consolidated": False,
        }
    )
    return ds

In [ ]:
# | export

def open_fits_as_dataset(fns):
    """Create a zarr reference from a set of fits files and open as a dask xarray Dataset.
    
    If you plan to reopen the Dataset repeatedly, then is would be better to create the
    JSON reference once with `create_zarr_ref_from_fits`, save it, and then just open
    the reference as needed with `open_zarr_ref_as_dataset`.
    """
    ref = create_zarr_ref_from_fits(fns)
    ds = open_zarr_ref_as_dataset(ref)
    return ds

## Example

In [ ]:
path = default_data_path("Q1_R1")
image_info = get_nisp_images_for_observation(2683, path=path)
fns = image_info.filename

In [ ]:
%%time
ref, wcs = create_zarr_ref_from_fits(fns)

In [ ]:
wcs

In [ ]:
WCS(wcs.sel(detector="DET11", dither=2, observation_id=2683).to_pandas())

In [ ]:
%%time
ds = open_zarr_ref_as_dataset(ref)

In [ ]:
ds

In [ ]:
median = ds['SCI'].median(dim=["observation_id", "dither", "filter"])

In [ ]:
%%time
median.compute()

In [ ]:
# | hide
import nbdev

nbdev.nbdev_export()